In [1]:
import pandas as pd
import yfinance as yf
from tqdm import tqdm
import time
import warnings

# Suppress yfinance's multi-threading-related FutureWarnings
warnings.simplefilter(action='ignore', category=FutureWarning)

# --- Configuration ---
INPUT_CSV = r"D:\Stock-Market-Indices\data\index_ticker_list_final.csv"
OUTPUT_CSV = "indices_verified_list.csv"

def verify_and_clean_index_list(input_file, output_file):
    """
    Reads a list of potential indices, verifies each ticker using yfinance,
    and saves only the confirmed indices to a new CSV file.
    """
    try:
        df = pd.read_csv(input_file)
    except FileNotFoundError:
        print(f"Error: Input file '{input_file}' not found. Please make sure it's in the correct directory.")
        return

    verified_indices = []
    
    print(f"Starting verification for {len(df)} entries from '{input_file}'...")

    for _, row in tqdm(df.iterrows(), total=df.shape[0], desc="Verifying Tickers"):
        index_name = row["Index Name"]
        ticker_str = row["Yahoo Finance Ticker"]

        if not isinstance(ticker_str, str) or pd.isna(ticker_str):
            print(f"\n[SKIP] Skipping row for '{index_name}' due to missing ticker.")
            continue
            
        try:
            # Fetch the info dictionary which contains metadata
            ticker_info = yf.Ticker(ticker_str).info
            
            # The most reliable way to check is the 'quoteType' key
            quote_type = ticker_info.get("quoteType")
            display_name = ticker_info.get("shortName") or ticker_info.get("longName") or index_name

            if quote_type == 'INDEX':
                tqdm.write(f"[KEEP] ✔️  '{ticker_str}' ({display_name}) is a valid INDEX.")
                verified_indices.append(row)
            elif quote_type == 'ETF':
                tqdm.write(f"[REMOVE] ❌ '{ticker_str}' ({display_name}) is an ETF.")
            else:
                # This handles other types like EQUITY, MUTUALFUND, CURRENCY etc.
                tqdm.write(f"[WARN] ⚠️  '{ticker_str}' ({display_name}) has an unexpected type: {quote_type}. Removing.")
            
            # Be polite to the API to avoid getting rate-limited
            time.sleep(0.05)

        except Exception as e:
            # This can happen for delisted, invalid, or restricted tickers
            tqdm.write(f"[ERROR] ❌ Could not fetch or process '{ticker_str}'. It may be invalid. Removing. Error: {e}")

    if not verified_indices:
        print("\nVerification complete. No valid indices were found.")
        return

    # Create a new DataFrame from the list of verified rows
    verified_df = pd.DataFrame(verified_indices)
    
    # Save the cleaned data to the new file
    verified_df.to_csv(output_file, index=False)
    
    print("\n" + "="*50)
    print("Verification Complete!")
    print(f"Original entries: {len(df)}")
    print(f"Kept {len(verified_df)} valid indices.")
    print(f"Removed {len(df) - len(verified_df)} entries (ETFs, invalid tickers, etc.).")
    print(f"Cleaned list saved to: '{output_file}'")
    print("="*50)


if __name__ == "__main__":
    verify_and_clean_index_list(INPUT_CSV, OUTPUT_CSV)

Starting verification for 82 entries from 'D:\Stock-Market-Indices\data\index_ticker_list_final.csv'...


Verifying Tickers:   1%|          | 1/82 [00:09<12:43,  9.43s/it]

[KEEP] ✔️  '^DJGT' (Dow Jones Global Titans 50 Inde) is a valid INDEX.


Verifying Tickers:   2%|▏         | 2/82 [00:12<07:36,  5.70s/it]

[REMOVE] ❌ 'VWRA.L' (VANGUARD FUNDS PLC VANGUARD FTS) is an ETF.


...

[KEEP] ✔️  '^HUI' (^HUI) is a valid INDEX.


Verifying Tickers:  99%|█████████▉| 81/82 [02:31<00:01,  1.10s/it]

[KEEP] ✔️  '^XAU' (PHLX Gold/Silver Sector) is a valid INDEX.


Verifying Tickers: 100%|██████████| 82/82 [02:32<00:00,  1.86s/it]

[REMOVE] ❌ 'PHO' (Invesco Water Resources ETF) is an ETF.

Verification Complete!
Original entries: 82
Kept 71 valid indices.
Removed 11 entries (ETFs, invalid tickers, etc.).
Cleaned list saved to: 'indices_verified_list.csv'


In [7]:
import pandas as pd
import yfinance as yf
from tqdm import tqdm
import time
import warnings

warnings.simplefilter(action='ignore', category=FutureWarning)

# --- Configuration ---
INPUT_CSV = "indices_verified_list.csv" 
OUTPUT_CSV = "indices_with_full_names.csv"

def get_best_name(ticker_info, original_name, ticker_str):
    """
    Determines the best possible name using a tiered logic.
    Returns the best name and the source it came from.
    """
    long_name = ticker_info.get('longName')
    short_name = ticker_info.get('shortName')
    
    # Priority 1: Use longName if it's descriptive
    if long_name and long_name.strip() and long_name != ticker_str:
        return long_name, "[longName]"
        
    # Priority 2: Use shortName if it's descriptive
    if short_name and short_name.strip() and short_name != ticker_str:
        return short_name, "[shortName]"
        
    # Priority 3: Fall back to the user-provided original name from the CSV
    # This is often better than a non-descriptive name from the API
    if original_name and original_name.strip() and original_name != ticker_str:
        return original_name, "[Original CSV Name]"
        
    # Last Resort: If even the original name is just the ticker, use shortName or longName
    # This handles cases where your CSV might have '^VIX' as the name.
    return short_name or long_name or original_name, "[Fallback]"


def add_full_names_to_csv(input_file, output_file):
    """
    Reads a CSV of verified indices, fetches the official full name for each ticker
    using yfinance with a robust tiered logic, and saves the result.
    """
    try:
        df = pd.read_csv(input_file)
    except FileNotFoundError:
        print(f"Error: Input file '{input_file}' not found.")
        return

    full_names_list = []
    
    print(f"Fetching full names for {len(df)} indices from '{input_file}'...")

    for _, row in tqdm(df.iterrows(), total=df.shape[0], desc="Normalizing Names"):
        original_name = row["Index Name"]
        ticker_str = row["Yahoo Finance Ticker"]
        best_name = original_name  # Default value
        source = "[Default]"

        try:
            ticker_info = yf.Ticker(ticker_str).info
            best_name, source = get_best_name(ticker_info, original_name, ticker_str)
            
            # Use a different log message for fallbacks
            if source in ["[Original CSV Name]", "[Fallback]"]:
                 tqdm.write(f"{source} ⚠️  Used fallback name for '{ticker_str}': '{best_name}'")
            else:
                 tqdm.write(f"{source} ✔️  Found name for '{ticker_str}': '{best_name}'")
            
            time.sleep(0.05)
        except Exception as e:
            best_name = original_name
            source = "[Error]"
            tqdm.write(f"{source} ❌ Error fetching '{ticker_str}', using original name. Error: {e}")

        full_names_list.append(best_name)

    df['Full Index Name'] = full_names_list
    df.rename(columns={'Index Name': 'Original Index Name'}, inplace=True)
    df = df[['Full Index Name', 'Yahoo Finance Ticker', 'Original Index Name']]

    df.to_csv(output_file, index=False)
    
    print("\n" + "="*50)
    print("Name Normalization Complete!")
    print(f"Processed {len(df)} indices.")
    print(f"New file with full names saved to: '{output_file}'")
    print("\nReview any '⚠️' warnings. You can improve them by editing your input CSV and re-running.")
    print("="*50)

if __name__ == "__main__":
    add_full_names_to_csv(INPUT_CSV, OUTPUT_CSV)

Fetching full names for 71 indices from 'indices_verified_list.csv'...


Normalizing Names:   1%|▏         | 1/71 [00:06<07:46,  6.67s/it]

[longName] ✔️  Found name for '^DJGT': 'Dow Jones Global Titans 50 Inde'


Normalizing Names:   3%|▎         | 2/71 [00:09<05:20,  4.65s/it]

[longName] ✔️  Found name for '^SPG100': 'S&P GLOBAL 100 ( C )'


...

[longName] ✔️  Found name for '^SOX': 'PHLX Semiconductor'


Normalizing Names:  99%|█████████▊| 70/71 [03:10<00:02,  2.27s/it]

[Original CSV Name] ⚠️  Used fallback name for '^HUI': 'HUI Gold Index (Metals)'


Normalizing Names: 100%|██████████| 71/71 [03:13<00:00,  2.73s/it]

[longName] ✔️  Found name for '^XAU': 'PHLX Gold/Silver Sector'

Name Normalization Complete!
Processed 71 indices.
New file with full names saved to: 'indices_with_full_names.csv'

Review any '⚠️' warnings. You can improve them by editing your input CSV and re-running.
